In [ ]:
!pip install ollama chromadb langchain 

In [ ]:
import os
import ollama
import chromadb
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [ ]:
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

# Load and split files
chunks = []
file_paths = [os.path.join("class_files", file) for file in os.listdir("class_files") if file.endswith('.rst')]

for file_path in file_paths:
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()
        chunks.extend(text_splitter.split_text(content))

print(f"Total chunks created: {len(chunks)}")

In [ ]:
client = chromadb.Client()
collection = client.create_collection(name="docs")

# store each document in a vector embedding database
for i, d in enumerate(chunks):
  response = ollama.embeddings(model="nomic-embed-text", prompt=d) # I've also tried "mxbai-embed-large" 
  embedding = response["embedding"]
  collection.add(
    ids=[str(i)],
    embeddings=[embedding],
    documents=[d]
  )

In [ ]:
# an example prompt
prompt = "Can you give me code to turn the servo to 45 degrees?"

# generate an embedding for the prompt and retrieve the most relevant doc
response = ollama.embeddings(
  prompt=prompt,
  model="nomic-embed-text" # note that this has to be the same model used to generate the embeddings 
)
results = collection.query(
  query_embeddings=[response["embedding"]],
  n_results=5
)  

In [ ]:
# store the data 
data = []

# iterate through the results and store the documents 
for doc_idx in range(len(results['ids'])):
    data.append(results['documents'][0][doc_idx].strip())

# store the data as a large string
data_str = "".join(data)

In [ ]:
# response from llama3 
output = ollama.generate(
  model="llama3", 
  prompt=f"Using this data: {data_str}; and, any information you believe is relevant to the user's question. Respond to this prompt: {prompt}"
)

print(output['response'])

In [ ]:
# response from mistral 
output = ollama.generate(
  model="mistral", 
  prompt=f"Using this data: {data_str}; and, any information you believe is relevant to the user's question. Respond to this prompt: {prompt}"
)

print(output['response'])

In [ ]:
# response from gemma:7b
output = ollama.generate(
  model="gemma:7b", 
  prompt=f"Using this data: {data_str}; and, any information you believe is relevant to the user's question. Respond to this prompt: {prompt}"
)

print(output['response'])

In [ ]:
# response from gemma:2b
output = ollama.generate(
  model="gemma:2b", 
  prompt=f"Using this data: {data_str}; and, any information you believe is relevant to the user's question. Respond to this prompt: {prompt}"
)

print(output['response'])